# Comparing the results of multiple 
This document 

## Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
from os import listdir
from os.path import isfile

Importing the collected data

In [3]:
from google.colab import files
uploaded = files.upload()
%ls

Saving binance-coin5m.csv to binance-coin5m (2).csv
Saving binance-coindaily.csv to binance-coindaily (1).csv
Saving binance-coinhourly.csv to binance-coinhourly (1).csv
Saving bitcoin-cash5m.csv to bitcoin-cash5m (1).csv
Saving bitcoin-cashdaily.csv to bitcoin-cashdaily (1).csv
Saving bitcoin-cashhourly.csv to bitcoin-cashhourly (1).csv
Saving bitcoin5m.csv to bitcoin5m (1).csv
Saving bitcoindaily.csv to bitcoindaily (1).csv
Saving bitcoinhourly.csv to bitcoinhourly (1).csv
Saving cardano5m.csv to cardano5m (1).csv
Saving cardanodaily.csv to cardanodaily (1).csv
Saving cardanohourly.csv to cardanohourly (1).csv
Saving chainlink5m.csv to chainlink5m (1).csv
Saving chainlinkdaily.csv to chainlinkdaily (1).csv
Saving chainlinkhourly.csv to chainlinkhourly (1).csv
Saving ethereum5m.csv to ethereum5m (1).csv
Saving ethereumdaily.csv to ethereumdaily (1).csv
Saving ethereumhourly.csv to ethereumhourly (1).csv
Saving litecoin5m.csv to litecoin5m (1).csv
Saving litecoindaily.csv to litecoinda

In [4]:
def plot_time_series(predicted, true, n_training, filename):
  """
  Plot the time series
  """
  plt.figure(figsize=(16,10)) #plotting
  plt.axvline(x=n_training, color="#ffd166", linestyle='-') #size of the training set

  plt.plot(predicted, label='Predicted Price', color="#118ab2") #predicted plot
  plt.plot(true, label='True Price', color="#06d6a0") #actual plot

  plt.title('Time-Series Prediction', fontsize=16)
  plt.xlabel('Time', fontsize=14)
  plt.ylabel('Price', fontsize=14)

  plt.xlim(0)
  plt.legend()
  plt.show()
  plt.savefig(filename) 

## LSTM Model


### Model Definition

In [17]:
class LSTMCustom(nn.Module):
    def __init__(self, num_classes, input_size, hidden_size, num_layers, seq_length):
        super(LSTMCustom, self).__init__()
        self.num_classes = num_classes #number of classes
        self.num_layers = num_layers #number of layers
        self.input_size = input_size #input size
        self.hidden_size = hidden_size #hidden state
        self.seq_length = seq_length #sequence length

        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True) #lstm
        self.fc_1 =  nn.Linear(hidden_size, 128) #fully connected 1
        self.fc = nn.Linear(128, num_classes) #fully connected last layer

        self.relu = nn.ReLU()
    
    def forward(self,x):
        h_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #hidden state
        c_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #internal state
        # Propagate input through LSTM
        output, (hn, cn) = self.lstm(x, (h_0, c_0)) #lstm with input, hidden, and internal state
        hn = hn.view(-1, self.hidden_size) #reshaping the data for Dense layer next
        out = self.relu(hn)
        out = self.fc_1(out) #first Dense
        out = self.relu(out) #relu
        out = self.fc(out) #Final Output
        return out

Model Parameters

In [6]:
num_epochs = 2000 #1000 epochs
learning_rate = 0.001 #0.001 lr

input_size = 22 #number of features
hidden_size = 40 #number of features in hidden state
num_layers = 1 #number of stacked lstm layers

num_classes = 1 #number of output classes 

In [9]:
criterion = torch.nn.MSELoss()  # mean-squared error for regression

### Training Each Crypto

In [10]:
mm = MinMaxScaler()
ss = StandardScaler()

In [12]:
def train_test_split_tensor(x, y):
  """
  Custom train/test splitting
  TODO: maybe shorten code by using sklearn.preprocessing.train_test_split
  """
  cutoff = round(x.shape[0] * 0.7)

  # split into train and test
  x_train = x[:cutoff, :]
  x_test = x[cutoff:,:]
  y_train = y[:cutoff, :]
  y_test = y[cutoff:, :]

  # convert to tensor
  x_train = Variable(torch.Tensor(x_train))
  x_test = Variable(torch.Tensor(x_test))
  y_train = Variable(torch.Tensor(y_train))
  y_test = Variable(torch.Tensor(y_test)) 
  
  return x_train, x_test, y_train, y_test, cutoff

In [20]:
def process_data(df):

  # shift the prices
  df2 = df.copy(deep=True)
  df['priceUsd'] = df.priceUsd.shift(1)
  df = df.iloc[1:-1]
  df2 = df2.iloc[1:-1]

  # split into x and y
  x = df.iloc[:, 0:-1]
  y = df2.iloc[:, 0:1]

  # transform the data
  x_ss = ss.fit_transform(x)
  y_mm = mm.fit_transform(y) 

  # get train and test split and convert to tensors
  x_train, x_test, y_train, y_test, cutoff = train_test_split_tensor(x_ss, y_mm)

  # reshape tensors 
  x_train = torch.reshape(x_train, (x_train.shape[0], 1, x_train.shape[1]))
  x_test = torch.reshape(x_test, (x_test.shape[0], 1, x_test.shape[1])) 

  return x_train, x_test, y_train, y_test, cutoff

In [24]:
def train(df, lstm):
  """
  Train the lstm
  """
  x_train, x_test, y_train, y_test, cutoff = process_data(df)

  for epoch in range(num_epochs):
    outputs = lstm.forward(x_train) #forward pass
    optimizer.zero_grad() #caluclate the gradient, manually setting to 0
  
    # obtain the loss function
    loss = criterion(outputs, y_train)
  
    loss.backward() #calculates the loss of the loss function
  
    optimizer.step() #improve from loss, i.e backprop
    if epoch % 100 == 0:
      print("Epoch: %d, loss: %1.5f" % (epoch, loss.item())) 

  return lstm, cutoff

In [ ]:
# dir_path = "data/"
# file_list = [f for f in listdir(dir_path) if isfile(join(dir_path, f))]
file_list = [f for f in listdir() if isfile(f)]

file_endings = ["5m.csv", "hourly.csv", "daily.csv"]

# FIXME: move to each fileending?
# im going to start with just 5m
file_ending = file_endings[0]
for filename in [name for name in file_list if file_ending in name]:
  crypto_name = filename.replace(file_ending, "")
  
  df = pd.read_csv(filename)
  # remove unused columns
  exclude_cols = [0,1,2,4]
  df = df.iloc[:, ~df.columns.isin(df.columns[exclude_cols])]

  # define LSTM class
  lstm = LSTMCustom(num_classes, input_size, hidden_size, num_layers, df.shape[1]) 
  optimizer = torch.optim.Adam(lstm.parameters(), lr=learning_rate) 

  lstm, cutoff = train(df, lstm)

  df_x_ss = ss.transform(df.iloc[:, 0:-1]) #old transformers
  df_y_mm = mm.transform(df.iloc[:, 0:1]) #old transformers

  df_x_ss = Variable(torch.Tensor(df_x_ss)) #converting to Tensors
  df_y_mm = Variable(torch.Tensor(df_y_mm))
  #reshaping the dataset
  df_x_ss = torch.reshape(df_x_ss, (df_x_ss.shape[0], 1, df_x_ss.shape[1]))

  train_predict = lstm(df_x_ss) #forward pass
  data_predict = train_predict.data.numpy() #numpy conversion
  dataY_plot = df_y_mm.data.numpy()

  data_predict = mm.inverse_transform(data_predict) #reverse transformation
  dataY_plot = mm.inverse_transform(dataY_plot)

  plot_name = f"{crypto_name}.png"
  plot_time_series(data_predict, dataY_plot, cutoff, plot_name)

  model_name = f"{crypto_name}.pth"
  torch.save(lstm.state_dict(), model_name)